# MedNexus — Azure AI Search Index Setup

This notebook creates the **`mednexus-clinical`** hybrid search index in Azure AI Search.

The index supports:
- **Keyword search** — OData filters by `patient_id`, full-text search on clinical notes
- **Vector search** — Semantic similarity via `text-embedding-3-small` (1536 dims, HNSW, cosine)
- **Hybrid search** — Combined keyword + vector for best retrieval quality

### Prerequisites
1. Azure AI Search resource (Basic tier or higher for vector support)
2. Azure OpenAI with `text-embedding-3-small` deployed
3. A `.env` file in the repo root with your credentials

## 1. Install Dependencies

In [ ]:
%pip install azure-search-documents==11.6.0b8 azure-identity openai python-dotenv --quiet

## 2. Load Configuration from `.env`

In [ ]:
import os
from dotenv import load_dotenv

# Load from the repo root .env
load_dotenv(dotenv_path="../.env", override=True)

SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT", "")
SEARCH_KEY = os.getenv("AZURE_SEARCH_KEY", "")
INDEX_NAME = os.getenv("AZURE_SEARCH_INDEX", "mednexus-clinical")

OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "")
OPENAI_KEY = os.getenv("AZURE_OPENAI_API_KEY", "")
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = 1536

print(f"Search Endpoint : {SEARCH_ENDPOINT}")
print(f"Index Name      : {INDEX_NAME}")
print(f"OpenAI Endpoint : {OPENAI_ENDPOINT}")
print(f"Embedding Model : {EMBEDDING_DEPLOYMENT} ({EMBEDDING_DIMENSIONS} dims)")

assert SEARCH_ENDPOINT, "Set AZURE_SEARCH_ENDPOINT in .env"
assert SEARCH_KEY, "Set AZURE_SEARCH_KEY in .env"

## 3. Define the Index Schema

| Field | Type | Purpose |
|-------|------|---------|
| `id` | String (key) | Unique document ID |
| `patient_id` | String | Filterable — scopes queries to one patient |
| `content_type` | String | `"xray"`, `"note"`, `"interview"`, `"ecg"`, `"lab"` |
| `source_agent` | String | Which agent produced this record |
| `content` | String | Raw text (notes, transcripts, AI-generated X-ray descriptions) |
| `analysis_summary` | String | Agent's structured summary |
| `content_vector` | Vector (1536) | Embedding of `content` for semantic search |
| `metadata_storage_path` | String | Link to original file in Blob / MCP |
| `timestamp` | DateTimeOffset | When this record was created |

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticSearch,
    SemanticPrioritizedFields,
    SemanticField,
)

# ── Fields ────────────────────────────────────────────────

fields = [
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
        filterable=True,
    ),
    SimpleField(
        name="patient_id",
        type=SearchFieldDataType.String,
        filterable=True,
        facetable=True,
    ),
    SimpleField(
        name="content_type",
        type=SearchFieldDataType.String,
        filterable=True,
        facetable=True,
    ),
    SimpleField(
        name="source_agent",
        type=SearchFieldDataType.String,
        filterable=True,
    ),
    SearchableField(
        name="content",
        type=SearchFieldDataType.String,
        analyzer_name="en.microsoft",
    ),
    SearchableField(
        name="analysis_summary",
        type=SearchFieldDataType.String,
        analyzer_name="en.microsoft",
    ),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBEDDING_DIMENSIONS,
        vector_search_profile_name="mednexus-vector-profile",
    ),
    SimpleField(
        name="metadata_storage_path",
        type=SearchFieldDataType.String,
        filterable=False,
    ),
    SimpleField(
        name="timestamp",
        type=SearchFieldDataType.DateTimeOffset,
        filterable=True,
        sortable=True,
    ),
]

print(f"Defined {len(fields)} fields")
for f in fields:
    vinfo = f" (vector {EMBEDDING_DIMENSIONS}d)" if f.name == "content_vector" else ""
    print(f"  • {f.name:30s} {str(f.type):40s}{vinfo}")

## 4. Configure Vector Search (HNSW + Cosine)

In [ ]:
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="mednexus-hnsw",
            parameters={
                "m": 4,            # Bi-directional links per node
                "efConstruction": 400,  # Build-time accuracy (higher = better index)
                "efSearch": 500,        # Query-time accuracy
                "metric": "cosine",     # Standard for text embeddings
            },
        ),
    ],
    profiles=[
        VectorSearchProfile(
            name="mednexus-vector-profile",
            algorithm_configuration_name="mednexus-hnsw",
        ),
    ],
)

print("Vector Search configured:")
print(f"  Algorithm : HNSW (cosine)")
print(f"  Profile   : mednexus-vector-profile")
print(f"  Dimensions: {EMBEDDING_DIMENSIONS}")

## 5. Configure Semantic Search (Re-Ranking)

Semantic search re-ranks keyword results using a Microsoft-trained language model.
This gives us the best of both worlds: fast vector recall + precise semantic re-ranking.

In [ ]:
semantic_config = SemanticConfiguration(
    name="mednexus-semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        content_fields=[SemanticField(field_name="content")],
        title_fields=[SemanticField(field_name="analysis_summary")],
        keywords_fields=[SemanticField(field_name="content_type")],
    ),
)

semantic_search = SemanticSearch(configurations=[semantic_config])
print("Semantic re-ranking configured: mednexus-semantic-config")

## 6. Create (or Update) the Index

In [ ]:
index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)

index_client = SearchIndexClient(
    endpoint=SEARCH_ENDPOINT,
    credential=AzureKeyCredential(SEARCH_KEY),
)

# create_or_update is idempotent — safe to re-run
result = index_client.create_or_update_index(index)
print(f"✅ Index '{result.name}' created/updated successfully!")
print(f"   Fields: {len(result.fields)}")
print(f"   Vector profiles: {len(result.vector_search.profiles)}")
print(f"   Semantic configs: {len(result.semantic_search.configurations)}")

In [ ]:
%pip install --upgrade openai --quiet

## 7. Test: Generate an Embedding

Verify the Azure OpenAI embedding deployment works before the agents need it.

In [ ]:

from openai import AzureOpenAI

openai_client = AzureOpenAI(
    azure_endpoint=OPENAI_ENDPOINT,
    api_key=OPENAI_KEY,
    api_version="2024-06-01",
)

test_text = "Patient reports chest fluttering when lying down, 3 weeks post valve replacement."

response = openai_client.embeddings.create(
    input=test_text,
    model=EMBEDDING_DEPLOYMENT,
)

embedding = response.data[0].embedding
print(f"✅ Embedding generated successfully!")
print(f"   Dimensions: {len(embedding)}")
print(f"   First 5 values: {embedding[:5]}")

## 8. Test: Index a Sample Document

Push one of our Patient 001 transcript records into the index to verify the full pipeline.

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from datetime import datetime, timezone

sample_content = (
    "Patient states: 'I feel fine, just a little tired. No chest pain really, "
    "maybe just a bit of a cough from the cold weather.' "
    "Vital signs: BP 138/88, HR 82, Temp 99.2°F, SpO2 96%. "
    "55 y/o male, history of mild asthma."
)

# Generate embedding for the content
emb_response = openai_client.embeddings.create(
    input=sample_content,
    model=EMBEDDING_DEPLOYMENT,
)
content_vector = emb_response.data[0].embedding

sample_doc = {
    "id": "P001-interview-001",
    "patient_id": "P001",
    "content_type": "interview",
    "source_agent": "patient_historian",
    "content": sample_content,
    "analysis_summary": "Patient downplaying symptoms. Reports tiredness and minor cough, denies chest pain. Vitals show mild hypertension and low-grade fever.",
    "content_vector": content_vector,
    "metadata_storage_path": "data/mock/P001_interview_transcript.txt",
    "timestamp": datetime.now(timezone.utc).isoformat(),
}

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=AzureKeyCredential(SEARCH_KEY),
)

result = search_client.upload_documents(documents=[sample_doc])
print(f"✅ Document indexed: {sample_doc['id']}")
print(f"   Succeeded: {result[0].succeeded}")

## 9. Test: Hybrid Search (Keyword + Vector)

Run a hybrid query that combines keyword matching with vector similarity.
This is exactly how the Patient Historian agent will query the index.

In [ ]:
from azure.search.documents.models import VectorizedQuery

# The query: a doctor asking about respiratory symptoms
query_text = "patient with respiratory symptoms and fever"

# Generate query embedding
q_response = openai_client.embeddings.create(
    input=query_text,
    model=EMBEDDING_DEPLOYMENT,
)
query_vector = q_response.data[0].embedding

# Hybrid search: keyword + vector, filtered to patient P001
results = search_client.search(
    search_text=query_text,
    vector_queries=[
        VectorizedQuery(
            vector=query_vector,
            k_nearest_neighbors=5,
            fields="content_vector",
        ),
    ],
    filter="patient_id eq 'P001'",
    select=["id", "patient_id", "content_type", "analysis_summary", "content"],
    top=5,
)

print(f"🔍 Hybrid search: '{query_text}'")
print(f"   Filter: patient_id eq 'P001'")
print(f"{'─' * 60}")
for i, doc in enumerate(results):
    print(f"\n   Result {i+1} (score: {doc['@search.score']:.4f})")
    print(f"   ID:      {doc['id']}")
    print(f"   Type:    {doc['content_type']}")
    print(f"   Summary: {doc['analysis_summary'][:100]}...")

## 10. Verify Index in Portal

You can also verify the index was created correctly:

In [ ]:
# List all indexes to confirm
for idx in index_client.list_indexes():
    field_count = len(idx.fields)
    vector_fields = [f.name for f in idx.fields if hasattr(f, 'vector_search_dimensions') and f.vector_search_dimensions]
    print(f"📋 Index: {idx.name}")
    print(f"   Fields: {field_count} ({len(vector_fields)} vector)")
    print(f"   Vector fields: {vector_fields}")
    print()

: 

---

## ✅ Done!

The `mednexus-clinical` index is ready. The agents in the MedNexus pipeline will:

1. **Patient Historian** → Extracts text from PDFs/audio, generates embeddings, pushes documents into this index
2. **Vision Specialist** → Generates X-ray analysis text, embeds it, pushes to the index
3. **Diagnostic Synthesis** → Queries this index with hybrid search to find all relevant data for cross-modality comparison

### Index Architecture
```
Patient File Upload
       │
       ▼
  Agent Pipeline ──embed──► content_vector (1536 dims)
       │                         │
       ▼                         ▼
  content (text) ──────► Azure AI Search Index
                         ├─ Keyword search (BM25)
                         ├─ Vector search (HNSW/cosine)
                         └─ Hybrid = best of both
```